# HCDE notebook D: the same method on a different base model (Qwen/Qwen3-8B)

Trains on exactly the v2 data (same settings as v2), so the only difference from v2 is the base model.
**GPU 0 trains while GPU 1 runs the base-model baselines** (base + one-line tag, base + full rubric), so the
comparison is base vs fine-tuned *on the same base model*. Then the fine-tuned model is evaluated on both GPUs,
everything is judged and the model is exported for Ollama. Test sets: IT and public benchmarks (both models), plus the
pressure test for the fine-tuned model. About 7–8 hours.

**Settings:** Accelerator **GPU T4 x2** · Internet **on**.

Qwen3 is trained with plain ChatML (thinking off); its baselines use Qwen3's own template with thinking switched off.

**Add-ons → Secrets** (add each and **tick it** for this notebook):

| Secret | Value |
| --- | --- |
| `TEACHER_BASE_URL` | `https://api.groq.com/openai/v1` |
| `TEACHER_API_KEY` | your Groq key(s), comma-separated |
| `TEACHER_MODEL` | `openai/gpt-oss-120b` |
| `JUDGE_MODEL` | `qwen/qwen3.8-27b` (same as in your `.env`) |
| `JUDGE_EXTRA` | `{"reasoning_effort": "none"}` |
| `JUDGE_MAX_TOKENS` | `600` |
| `GITHUB_TOKEN` | only if the AnchorAI repo is private |

**How to run:** **Save Version → Save & Run All (Commit)**, then download from the version's Output:
`hcde_qwen3/hcde-qwen3-results.zip` and `hcde_qwen3/gguf/` (GGUF + Modelfile).

In [ ]:
BASE = 'Qwen/Qwen3-8B'
NAME = 'qwen3'          # condition label: tuned_<NAME>
CHATML = '--chatml'

import os, subprocess, time, glob, json, shutil
WORK = "/kaggle/working"
REPO = f"{WORK}/AnchorAI"
BRANCH = "dev/dulanja"
OUTDIR = f"{WORK}/hcde_qwen3"          # everything worth keeping ends up here
LOGS = f"{OUTDIR}/logs"
os.makedirs(LOGS, exist_ok=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()
def secret(name):
    try:
        return _secrets.get_secret(name)
    except Exception:
        return ""

token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/DulanjaR/AnchorAI.git" if token else "https://github.com/DulanjaR/AnchorAI.git"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

loaded = [k for k in ["TEACHER_BASE_URL", "TEACHER_API_KEY", "TEACHER_MODEL", "TEACHER_EXTRA",
                      "JUDGE_MODEL", "JUDGE_EXTRA", "JUDGE_MAX_TOKENS", "HF_TOKEN"] if secret(k)]
for k in loaded:
    os.environ[k] = secret(k)
HAVE_API = "TEACHER_BASE_URL" in loaded and "TEACHER_API_KEY" in loaded
print("secrets loaded:", loaded or "none")
if not HAVE_API:
    print("WARNING: no Groq secrets -> no judging (replies are still saved)")

subprocess.run("pip install -q -U peft bitsandbytes accelerate && pip uninstall -q -y torchao", shell=True)
GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
print("GPUs:", GPUS, "(both are used in parallel)" if GPUS > 1 else "(steps run one after another)")

PY = "python -m components.c4_hcde.training"
DATA = "components/c4_hcde/training/data/dataset"

def start(name, cmds, gpu=None):
    """Run shell commands one after another in the background (a failed command does not stop the
    next one). Output goes to logs/<name>.log."""
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    log = open(f"{LOGS}/{name}.log", "w")
    p = subprocess.Popen(" ; ".join(cmds), shell=True, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT)
    p.name = name
    print(f"started {name}" + (f" on GPU {gpu}" if gpu is not None else ""))
    return p

def tail(name, n=2):
    try:
        lines = [l.rstrip() for l in open(f"{LOGS}/{name}.log", errors="replace").read().replace("\r", "\n").splitlines() if l.strip()]
        return " | ".join(lines[-n:])[-300:]
    except FileNotFoundError:
        return ""

def wait(procs, every=300):
    """Block until all background jobs finish, printing progress every few minutes."""
    t0 = last = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(10)
        if time.time() - last < every and any(p.poll() is None for p in procs):
            continue
        last = time.time()
        print(f"--- {int((time.time() - t0) / 60)} min")
        for p in procs:
            print(f"  {p.name}: {'running' if p.poll() is None else 'done'} :: {tail(p.name)}", flush=True)
    for p in procs:
        print(f"{p.name}: exit code {p.returncode}  :: {tail(p.name, 3)}")

def run_on_gpus(chains):
    """chains = list of (name, [commands]); one chain per GPU, in parallel when there are 2 GPUs."""
    if GPUS > 1:
        wait([start(n, c, gpu=i % GPUS) for i, (n, c) in enumerate(chains)])
    else:
        for n, c in chains:
            wait([start(n, c, gpu=0)])

In [ ]:
EXTRA = [f for f in [f"{DATA}/public_train.jsonl", f"{DATA}/aug_shared.jsonl", f"{DATA}/aug_probe.jsonl",
                     f"{DATA}/aug_pressure.jsonl"] if os.path.exists(f)]
print("extra training files:", EXTRA)
RUN = f"{OUTDIR}/run"
TR = f"--test {DATA}/transfer_raw.jsonl --elements {DATA}/transfer_elements.json"
PB = f"--test {DATA}/public_test.jsonl --elements {DATA}/public_elements.json"
B = f"--base-model {BASE} --conditions base_tag base_rubric --batch-size 16"
# baselines: IT + public test sets; the fine-tuned model also gets the pressure test (time budget on a T4)
BASE_SETS = [("it", ""), ("public", PB)]
SETS = [("it", ""), ("public", PB), ("it-pressure", "--pressure")]

train = [f"{PY}.train_qlora --base-model {BASE} --out {RUN} --epochs 2 {CHATML} --extra-train " + " ".join(EXTRA)]
baselines = [f"{PY}.evaluate {B} {opts} --out {OUTDIR}/base-{s}" for s, opts in BASE_SETS]
if GPUS > 1:
    wait([start("train", train, gpu=0), start("baselines", baselines, gpu=1)])
else:
    wait([start("train", train, gpu=0)]); wait([start("baselines", baselines, gpu=0)])
try:
    m = json.load(open(f"{RUN}/training_meta.json"))
    print("best eval loss:", m["eval"].get("eval_loss"), "| minutes:", m["train_minutes"])
except Exception as e:
    print("TRAINING DID NOT FINISH, see logs/train.log:", e, "\n", tail("train", 15))

In [ ]:
T = f"--base-model {BASE} --adapter {RUN}/adapter --conditions tuned_tag --tuned-name tuned_{NAME} --batch-size 16"
cmds = {s: f"{PY}.evaluate {T} {opts} --out {OUTDIR}/tuned-{s}" for s, opts in SETS}
run_on_gpus([("tuned_a", [cmds["public"]]), ("tuned_b", [cmds["it"], cmds["it-pressure"]])])

In [ ]:
subprocess.run("apt-get -qq install -y cmake > /dev/null", shell=True)
export = start("export_gguf", [f"{PY}.export_gguf --base-model {BASE} --adapter {RUN}/adapter --out {OUTDIR}/gguf "
                               "--llama-cpp /tmp/llama.cpp --work-dir /tmp/hcde-export"])
EL = {"it": "solution_elements.json", "it-pressure": "solution_elements.json",
      "transfer": "transfer_elements.json", "public": "public_elements.json"}

if HAVE_API:
    jobs = []
    for d, elements in [(f"{OUTDIR}/{kind}-{s}", f"{DATA}/{EL[s]}") for kind, sets in (("base", BASE_SETS), ("tuned", SETS)) for s, _ in sets]:
        if os.path.exists(f"{d}/generations.jsonl"):
            name = "judge_" + d.rstrip("/").split("/")[-1]
            # run twice: the second pass only re-scores rows the first one could not (rate limits)
            cmd = f"{PY}.evaluate --from-generations {d}/generations.jsonl --elements {elements} --out {d} --judge"
            jobs.append(start(name, [cmd, cmd]))
    wait(jobs)
else:
    print("no Groq secrets: replies are saved, judge them later on the Mac with evaluate --from-generations ... --judge")

wait([export])
subprocess.run(f"{PY}.stats "
               f"--set it {OUTDIR}/base-it/generations.jsonl {OUTDIR}/tuned-it/generations.jsonl "
               f"{OUTDIR}/base-it-pressure/generations.jsonl {OUTDIR}/tuned-it-pressure/generations.jsonl "
               f"--set public {OUTDIR}/base-public/generations.jsonl {OUTDIR}/tuned-public/generations.jsonl "
               f"--out {OUTDIR}/stats.md", shell=True)
print(open(f"{OUTDIR}/stats.md").read() if os.path.exists(f"{OUTDIR}/stats.md") else "no stats")
os.chdir(OUTDIR)
subprocess.run(f"zip -qr hcde-{NAME}-results.zip run/adapter run/training_meta.json base-* tuned-* stats.md logs", shell=True)
os.chdir(REPO)
for f in sorted(glob.glob(f"{OUTDIR}/**", recursive=True)):
    if os.path.isfile(f) and f.endswith((".zip", ".gguf", "Modelfile")):
        print(f"{os.path.getsize(f) / 1e6:9.1f} MB  {f}")